In [ ]:
import os
import json
import platform
from datetime import datetime
from collections import deque

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_ACCEPTED_COUNTRIES = 150
MAX_COUNTRY_CANDIDATES = 50000
PSI_GRID_N = 200
PSI0_GRID_N = 21

T_FIXED = 200.0
DT = 0.05

CHI_RANGE = (1e-1, 1e2)
SIGMA_RANGE = (1e-3, 1e0)
OMEGA_MULT_RANGE = (0.5, 2.0)
MU_RANGE = (0.01, 0.1)
MU_DESIGNS_TO_RUN = ("baseline_prior", "threshold_targeted")
THRESHOLD_TARGET_U_RANGE = (0.1, 0.9)

MC_NAME = "mc_block2_threshold_dynamics"
RUN_TS = datetime.now().strftime("%y%m%d_%H%M")
BASE_OUT_DIR = os.getcwd()
OUT_DIR = os.path.join(BASE_OUT_DIR, f"{MC_NAME}_{RUN_TS}")
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 20260212

PROGRESS_EVERY_CANDIDATES = 500
PROGRESS_EVERY_ACCEPTED = 10
PROGRESS_EVERY_COUNTRIES = 10

MC_CONFIG = {
    "N_COUNTRIES": N_ACCEPTED_COUNTRIES,
    "psi_grid_n": PSI_GRID_N,
    "psi0_grid_n": PSI0_GRID_N,
    "rho_bar_range": (1.0, 5.0),
    "a_range": (0.5, 2.0),
    "c_lambda_range": (0.1, 1.0),
    "eta_lambda_range": (1.1, 2.0),
    "c_ell_range": (0.05, 0.5),
    "eta_ell_range": (1.1, 2.0),
    "psi_L_range": (0.0, 0.3),
    "Gamma_range": (0.01, 0.05),
    "k_range": (0.1, 0.5),
    "s_N_range": (0.05, 0.5),
    "kappa_c_range": (0.05, 2.0),
    "chi_range": CHI_RANGE,
    "sigma_range": SIGMA_RANGE,
    "omega_mult_range": OMEGA_MULT_RANGE,
    "mu_range": MU_RANGE,
    "max_bad_share": 0.05,
    "eps_solv": 1e-12,
    "tol_g": 1e-12,
    "tol_mono": 1e-10,
    "DERIV_RHO_MIN": 1e-8,
    "OMEGA_MIN": 1e-8,
    "ratio_bins_log10": [-3, -2, -1, 0, 1, 2, 3, 4, 5],
}

SIM_CONFIG = {
    "T_fixed": T_FIXED,
    "dt": DT,
    "eps_psi": 1e-4,
    "eps_g": 1e-4,
    "stable_window": 200,
}

IC_NEAR_EPS = 0.05
IC_FAR_BELOW_MULT = 0.50
IC_FAR_ABOVE_MULT = 1.00
IC_FAR_ABOVE_CAP_MULT = 0.50

def g_scale(gstar):
    return 1.0 + abs(gstar)

def save_df(df, tag):
    path = os.path.join(OUT_DIR, f"{MC_NAME}_{tag}_{RUN_TS}.csv")
    df.to_csv(path, index=False)
    return path

def mono_nondec(x, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if not np.all(np.isfinite(x)):
        return False
    return bool(np.all(np.diff(x) >= -tol))

def mono_noninc(x, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if not np.all(np.isfinite(x)):
        return False
    return bool(np.all(np.diff(x) <= tol))

def r_func(rho, a, b):
    return a * rho - b * rho**2

def r_prime(rho, a, b):
    return a - 2.0 * b * rho

def r_second(b):
    return -2.0 * b

def lam_func(rho, c_lam, eta_lam):
    return 1.0 - np.exp(-c_lam * rho**eta_lam)

def lam_prime(rho, c_lam, eta_lam):
    arr = np.asarray(rho, dtype=float)
    out = np.zeros_like(arr)
    valid = arr >= MC_CONFIG["DERIV_RHO_MIN"]
    if np.any(valid):
        v = arr[valid]
        out[valid] = np.exp(-c_lam * v**eta_lam) * (c_lam * eta_lam * v**(eta_lam - 1.0))
    return out if arr.ndim > 0 else float(out)

def lam_second(rho, c_lam, eta_lam, DERIV_RHO_MIN):
    arr = np.asarray(rho, dtype=float)
    out = np.zeros_like(arr)
    valid = arr >= DERIV_RHO_MIN
    if np.any(valid):
        v = arr[valid]
        f = np.exp(-c_lam * v**eta_lam)
        g = c_lam * eta_lam * v**(eta_lam - 1.0)
        gp = c_lam * eta_lam * (eta_lam - 1.0) * v**(eta_lam - 2.0)
        out[valid] = f * (gp - g**2)
    return out if arr.ndim > 0 else float(out)

def ell_func(rho, c_ell, eta_ell):
    return c_ell * rho**eta_ell

def ell_prime(rho, c_ell, eta_ell):
    arr = np.asarray(rho, dtype=float)
    out = np.zeros_like(arr)
    valid = arr >= MC_CONFIG["DERIV_RHO_MIN"]
    if np.any(valid):
        v = arr[valid]
        out[valid] = c_ell * eta_ell * v**(eta_ell - 1.0)
    return out if arr.ndim > 0 else float(out)

def ell_second(rho, c_ell, eta_ell, DERIV_RHO_MIN):
    arr = np.asarray(rho, dtype=float)
    out = np.zeros_like(arr)
    valid = arr >= DERIV_RHO_MIN
    if np.any(valid):
        v = arr[valid]
        out[valid] = c_ell * eta_ell * (eta_ell - 1.0) * v**(eta_ell - 2.0)
    return out if arr.ndim > 0 else float(out)

def B_func(rho, c_lam, eta_lam, c_ell, eta_ell):
    return lam_prime(rho, c_lam, eta_lam) * ell_func(rho, c_ell, eta_ell) + lam_func(rho, c_lam, eta_lam) * ell_prime(rho, c_ell, eta_ell)

def B_prime(rho, c_lam, eta_lam, c_ell, eta_ell, DERIV_RHO_MIN):
    lam = lam_func(rho, c_lam, eta_lam)
    lam_p = lam_prime(rho, c_lam, eta_lam)
    lam_pp = lam_second(rho, c_lam, eta_lam, DERIV_RHO_MIN)
    ell = ell_func(rho, c_ell, eta_ell)
    ell_p = ell_prime(rho, c_ell, eta_ell)
    ell_pp = ell_second(rho, c_ell, eta_ell, DERIV_RHO_MIN)
    return lam_pp * ell + 2.0 * lam_p * ell_p + lam * ell_pp

def solve_rho_star(psi_val, rho_bar, a, b, c_lam, eta_lam, c_ell, eta_ell, tol=1e-12, max_iter=300):
    def F(rho):
        return r_prime(rho, a, b) - (1.0 - psi_val) * B_func(rho, c_lam, eta_lam, c_ell, eta_ell)

    F0 = r_prime(0.0, a, b) - (1.0 - psi_val) * B_func(0.0, c_lam, eta_lam, c_ell, eta_ell)
    Fhi = r_prime(rho_bar, a, b) - (1.0 - psi_val) * B_func(rho_bar, c_lam, eta_lam, c_ell, eta_ell)

    if not np.isfinite(Fhi):
        return np.nan, "nonfinite"

    if abs(F0) <= tol:
        return 0.0, "corner_low"
    if abs(Fhi) <= tol:
        return rho_bar, "corner_high"

    if F0 * Fhi < 0.0:
        lo, hi = 0.0, rho_bar
        for _ in range(max_iter):
            mid = 0.5 * (lo + hi)
            f_mid = F(mid)
            if not np.isfinite(f_mid):
                return np.nan, "nonfinite"
            if abs(f_mid) < tol or (hi - lo) < tol:
                return mid, "interior"
            if np.sign(f_mid) == np.sign(F0):
                lo = mid
            else:
                hi = mid
        return np.nan, "no_converge"

    if F0 > 0.0 and Fhi > 0.0:
        return rho_bar, "corner_high"
    if F0 < 0.0 and Fhi < 0.0:
        return 0.0, "corner_low"

    return np.nan, "no_bracket"

def check_kkt_status(psi_val, rho, status, a, b, c_lam, eta_lam, c_ell, eta_ell, tol_kkt=1e-10):
    F_val = r_prime(rho, a, b) - (1.0 - psi_val) * B_func(rho, c_lam, eta_lam, c_ell, eta_ell)
    if status == "interior":
        return abs(F_val) <= tol_kkt
    elif status == "corner_low":
        return F_val <= tol_kkt
    elif status == "corner_high":
        return F_val >= -tol_kkt
    return False

def draw_primitives(rng_local, cfg):
    return {
        "rho_bar": rng_local.uniform(*cfg["rho_bar_range"]),
        "a": rng_local.uniform(*cfg["a_range"]),
        "c_lambda": rng_local.uniform(*cfg["c_lambda_range"]),
        "eta_lambda": rng_local.uniform(*cfg["eta_lambda_range"]),
        "c_ell": rng_local.uniform(*cfg["c_ell_range"]),
        "eta_ell": rng_local.uniform(*cfg["eta_ell_range"]),
        "psi_L": rng_local.uniform(*cfg["psi_L_range"]),
        "Gamma": rng_local.uniform(*cfg["Gamma_range"]),
        "k": rng_local.uniform(*cfg["k_range"]),
        "s_N": rng_local.uniform(*cfg["s_N_range"]),
        "kappa_c": rng_local.uniform(*cfg["kappa_c_range"]),
    }

def draw_admissible_primitives(rng_local, cfg, max_attempts=10000):
    for attempt in range(max_attempts):
        theta = draw_primitives(rng_local, cfg)
        rho_bar = theta["rho_bar"]
        c_ell = theta["c_ell"]
        eta_ell = theta["eta_ell"]
        k = theta["k"]

        ell_bar = ell_func(rho_bar, c_ell, eta_ell)
        if np.isfinite(ell_bar) and (0.0 < ell_bar < 1.0) and (k * ell_bar < 1.0):
            theta["b"] = theta["a"] / (2.0 * rho_bar)
            theta["r0"] = 0.0
            return theta
    raise RuntimeError(
        f"Failed to generate admissible primitives after {max_attempts} attempts. "
        "Verify that config draw ranges allow ell(rho_bar) < 1 and k*ell(rho_bar) < 1."
    )

def g_star_stationary(psi, Gamma_eff, kappa_c):
    if (not np.isfinite(psi)) or (not np.isfinite(Gamma_eff)) or (not np.isfinite(kappa_c)):
        return np.nan
    if Gamma_eff <= 0.0:
        return 0.0
    if psi <= 0.0:
        return Gamma_eff
    disc = 1.0 + 4.0 * kappa_c * psi * Gamma_eff
    if disc <= 0.0:
        return 0.0
    return (-1.0 + np.sqrt(disc)) / (2.0 * kappa_c * psi)

def build_manifold_objects(theta, cfg):
    DERIV_RHO_MIN = cfg["DERIV_RHO_MIN"]

    rho_bar, a, b = theta["rho_bar"], theta["a"], theta["b"]
    c_lam, eta_lam = theta["c_lambda"], theta["eta_lambda"]
    c_ell, eta_ell = theta["c_ell"], theta["eta_ell"]
    psi_L, s_N, Gamma, k, kappa_c = theta["psi_L"], theta["s_N"], theta["Gamma"], theta["k"], theta["kappa_c"]

    psi_grid = np.linspace(psi_L, 1.0, cfg["psi_grid_n"])

    rho0, st0 = solve_rho_star(0.0, rho_bar, a, b, c_lam, eta_lam, c_ell, eta_ell)
    base_ok = st0 in {"interior", "corner_low", "corner_high"}
    if not base_ok or (not np.isfinite(rho0)):
        n = psi_grid.size
        return {
            "psi_grid": psi_grid,
            "solver_ok_endpoints": False,
            "excluded_bad_share": True,
            "bad_share": 1.0,
            "valid_mask": np.zeros(n, dtype=bool),
            "rho_star": np.full(n, np.nan),
            "rK": np.full(n, np.nan),
            "Gamma_eff": np.full(n, np.nan),
            "g_star": np.full(n, np.nan),
            "W_star": np.full(n, np.nan),
            "ell": np.full(n, np.nan),
            "lam": np.full(n, np.nan),
            "solv_ok": False,
            "gstar_ok": False,
            "mono_ok": False,
            "deriv_ok": False,
            "omega_base": np.nan,
            "solver_status": np.array(["base_fail"] * n, dtype=object),
        }

    base_drag = k * lam_func(rho0, c_lam, eta_lam) * ell_func(rho0, c_ell, eta_ell)

    n = psi_grid.size
    rho_star = np.full(n, np.nan)
    solver_status = np.empty(n, dtype=object)
    solver_ok = np.zeros(n, dtype=bool)

    deriv_ok = True
    for i, psi in enumerate(psi_grid):
        rho_i, status = solve_rho_star(psi, rho_bar, a, b, c_lam, eta_lam, c_ell, eta_ell)
        solver_status[i] = status
        solver_ok[i] = status in {"interior", "corner_low", "corner_high"}
        rho_star[i] = rho_i if solver_ok[i] else np.nan

        if status == "interior":
            if (not np.isfinite(rho_i)) or (rho_i < DERIV_RHO_MIN):
                deriv_ok = False
            else:
                Bp = B_prime(rho_i, c_lam, eta_lam, c_ell, eta_ell, DERIV_RHO_MIN)
                if not np.isfinite(Bp):
                    deriv_ok = False

    solver_ok_endpoints = bool(solver_ok[0] and solver_ok[-1])

    lam = lam_func(rho_star, c_lam, eta_lam)
    ell = ell_func(rho_star, c_ell, eta_ell)
    drag = k * lam * ell - base_drag
    Gamma_eff = Gamma - drag
    rK = r_func(rho_star, a, b) - (1.0 - psi_grid) * lam * ell
    g_star = np.array([g_star_stationary(psi_grid[i], Gamma_eff[i], kappa_c) for i in range(n)])
    W_star = rK - g_star

    valid = solver_ok.copy()
    valid &= np.isfinite(lam) & np.isfinite(ell)
    valid &= np.isfinite(rK) & np.isfinite(Gamma_eff)
    valid &= np.isfinite(g_star) & np.isfinite(W_star)

    bad_share = 1.0 - float(np.mean(valid))
    excluded_bad_share = bool(bad_share > cfg["max_bad_share"])

    solv_grid = (s_N - psi_grid * k * ell) > (cfg["eps_solv"] * max(1.0, s_N))
    g_star_ok = np.isfinite(g_star) & (g_star >= -cfg["tol_g"])

    if np.any(valid) and solver_ok_endpoints and (not excluded_bad_share):
        solv_ok = bool(np.all(solv_grid[valid]))
        gstar_ok = bool(np.all(g_star_ok[valid]))
        mono_ok = bool(
            mono_nondec(rho_star[valid], tol=1e-7)
            and mono_nondec(drag[valid], tol=1e-9)
            and mono_noninc(g_star[valid], tol=1e-9)
            and mono_nondec(W_star[valid], tol=1e-9)
        )
    else:
        solv_ok = False
        gstar_ok = False
        mono_ok = False

    Wabs = np.abs(W_star[valid]) if np.any(valid) else np.array([])
    omega_base = float(np.median(Wabs)) if Wabs.size else np.nan
    if (not np.isfinite(omega_base)) or (omega_base <= 0.0):
        omega_base = np.nan
    else:
        omega_base = max(omega_base, cfg["OMEGA_MIN"])

    return {
        "psi_grid": psi_grid,
        "solver_ok_endpoints": solver_ok_endpoints,
        "excluded_bad_share": excluded_bad_share,
        "bad_share": float(bad_share),
        "valid_mask": valid,
        "rho_star": rho_star,
        "rK": rK,
        "Gamma_eff": Gamma_eff,
        "g_star": g_star,
        "W_star": W_star,
        "ell": ell,
        "lam": lam,
        "solv_ok": bool(solv_ok),
        "gstar_ok": bool(gstar_ok),
        "mono_ok": bool(mono_ok),
        "deriv_ok": bool(deriv_ok),
        "omega_base": omega_base,
        "solver_status": solver_status,
    }

def make_grid_lookups(objs, theta):
    psi_grid = objs["psi_grid"]
    valid = objs["valid_mask"]

    if not (valid[0] and valid[-1]):
        raise ValueError("Grid coverage fail: Endpoints are not valid.")

    x = psi_grid[valid]
    rK_arr = objs["rK"][valid]
    Ge_arr = objs["Gamma_eff"][valid]
    kappa_c = theta["kappa_c"]

    def rK_of_psi(psi):
        psi_clipped = float(np.clip(psi, x[0], x[-1]))
        return float(np.interp(psi_clipped, x, rK_arr))

    def Ge_of_psi(psi):
        psi_clipped = float(np.clip(psi, x[0], x[-1]))
        return float(np.interp(psi_clipped, x, Ge_arr))

    def gstar_of_psi(psi):
        return float(g_star_stationary(psi, Ge_of_psi(psi), kappa_c))

    return rK_of_psi, Ge_of_psi, gstar_of_psi

def classify_static_threshold(objs, mu, tol=1e-10):
    """Classify the quasi-static institutional field relative to the maintenance threshold."""
    valid = np.asarray(objs["valid_mask"], dtype=bool)

    if len(valid) == 0 or not (valid[0] and valid[-1]):
        return {
            "threshold_class": "unclassified",
            "W_floor": np.nan,
            "W_top": np.nan,
            "psi_threshold_hat": np.nan,
            "interior_threshold": False,
        }

    if not objs.get("mono_ok", False):
        W_star_raw = objs["W_star"]
        W_floor = float(W_star_raw[0]) if len(W_star_raw) else np.nan
        W_top = float(W_star_raw[-1]) if len(W_star_raw) else np.nan
        return {
            "threshold_class": "unclassified_nonmonotone",
            "W_floor": W_floor,
            "W_top": W_top,
            "psi_threshold_hat": np.nan,
            "interior_threshold": False,
        }

    psi_grid = np.asarray(objs["psi_grid"])[valid]
    W_star = np.asarray(objs["W_star"])[valid]

    W_floor = float(W_star[0])
    W_top = float(W_star[-1])

    if not (np.isfinite(W_floor) and np.isfinite(W_top) and np.isfinite(mu)):
        return {
            "threshold_class": "unclassified",
            "W_floor": W_floor,
            "W_top": W_top,
            "psi_threshold_hat": np.nan,
            "interior_threshold": False,
        }

    if W_floor >= mu - tol:
        threshold_class = "born_high"
        psi_threshold_hat = float(psi_grid[0]) if abs(W_floor - mu) <= tol else np.nan
        interior_threshold = False
    elif W_top <= mu + tol:
        threshold_class = "born_low"
        psi_threshold_hat = float(psi_grid[-1]) if abs(W_top - mu) <= tol else np.nan
        interior_threshold = False
    else:
        threshold_class = "bistable_interior"
        interior_threshold = True
        diff = W_star - mu
        cross_idx = np.where(diff[:-1] * diff[1:] <= 0.0)[0]
        psi_threshold_hat = np.nan

        if len(cross_idx) > 0:
            j = int(cross_idx[0])
            W_lo, W_hi = diff[j], diff[j + 1]
            p_lo, p_hi = psi_grid[j], psi_grid[j + 1]

            if abs(W_hi - W_lo) > tol:
                psi_threshold_hat = float(p_lo - W_lo * (p_hi - p_lo) / (W_hi - W_lo))
            else:
                psi_threshold_hat = float(p_lo)

    return {
        "threshold_class": threshold_class,
        "W_floor": W_floor,
        "W_top": W_top,
        "psi_threshold_hat": psi_threshold_hat,
        "interior_threshold": bool(interior_threshold),
    }

def riccati_g_step(g, Ge, a_coeff, chi, dt):
    """
    Exact positive-preserving Riccati step under the nonnegative extended-growth convention.
    For Ge < 0, the forcing is clipped to the decay-to-zero branch rather than the raw negative-forcing ODE.
    Ensures g_new >= 0 for all initial g >= 0.
    """
    if Ge <= 0.0:
        return g * np.exp(-chi * dt) / (1.0 + a_coeff * g * (1.0 - np.exp(-chi * dt)))

    if a_coeff < 1e-14:
        return Ge + (g - Ge) * np.exp(-chi * dt)

    disc = 1.0 + 4.0 * a_coeff * Ge
    if disc < 0.0:
        return g * np.exp(-chi * dt)

    sqrt_disc = np.sqrt(disc)
    g1 = (-1.0 + sqrt_disc) / (2.0 * a_coeff)
    g2 = (-1.0 - sqrt_disc) / (2.0 * a_coeff)

    sep = g1 - g2
    exp_factor = np.exp(-chi * a_coeff * sep * dt)

    denom_0 = g - g2
    if abs(denom_0) < 1e-30:
        return g1

    R0 = (g - g1) / denom_0
    R_new = R0 * exp_factor

    denom = 1.0 - R_new
    if abs(denom) < 1e-30:
        return g1

    return (g1 - R_new * g2) / denom

def compute_break_diagnostics_fixed(g_series, t_series):
    if len(g_series) < 20:
        return {
            "break_score_mean": np.nan,
            "break_score_var": np.nan,
            "t_break_hat": np.nan,
            "collapse_flag": False,
            "t_collapse_hat": np.nan,
        }

    g_arr = np.array(g_series)
    t_arr = np.array(t_series)
    n = len(g_arr)
    min_segment = max(5, n // 10)

    breaks_mean = []
    breaks_var = []
    break_times = []

    for split_idx in range(min_segment, n - min_segment):
        g_left = g_arr[:split_idx]
        g_right = g_arr[split_idx:]

        breaks_mean.append(abs(np.mean(g_left) - np.mean(g_right)))
        breaks_var.append(abs(np.var(g_left) - np.var(g_right)))
        break_times.append(t_arr[split_idx])

    if not breaks_mean:
        break_score_mean, break_score_var, t_break_hat = np.nan, np.nan, np.nan
    else:
        max_idx = int(np.argmax(breaks_mean))
        break_score_mean = float(breaks_mean[max_idx])
        break_score_var = float(breaks_var[max_idx])
        t_break_hat = float(break_times[max_idx])

    first_quarter = g_arr[:len(g_arr)//4]
    if len(first_quarter) > 0:
        q25, q75 = np.percentile(first_quarter, [25, 75])
        g_threshold = max(0.0, np.median(first_quarter) - 1.5 * (q75 - q25))
    else:
        g_threshold = 0.0

    below_threshold = g_arr < g_threshold
    collapse_flag = False
    t_collapse_hat = np.nan

    for i in range(len(below_threshold) - 5 + 1):
        if np.all(below_threshold[i:i+5]):
            collapse_flag = True
            t_collapse_hat = float(t_arr[i])
            break

    return {
        "break_score_mean": float(break_score_mean) if np.isfinite(break_score_mean) else np.nan,
        "break_score_var": float(break_score_var) if np.isfinite(break_score_var) else np.nan,
        "t_break_hat": float(t_break_hat) if np.isfinite(t_break_hat) else np.nan,
        "collapse_flag": bool(collapse_flag),
        "t_collapse_hat": float(t_collapse_hat) if np.isfinite(t_collapse_hat) else np.nan,
    }

def rhs_psi(psiP, g, psi_L, rK_of_psi, omega, sigma, mu):
    psi = psi_L + psiP
    W = rK_of_psi(psi) - g - mu
    return sigma * np.tanh(W / omega)

def rhs_1d(psiP, psi_L, rK_of_psi, gstar_of_psi, sigma, omega, mu):
    psi = psi_L + psiP
    Wstar = rK_of_psi(psi) - gstar_of_psi(psi) - mu
    return sigma * np.tanh(Wstar / omega)

def classify_stable_tail(psi_tail, g_tail, psi_L, gstar_of_psi, eps_psi, eps_g):
    """Shared basin detector for both 2D and quasi-static 1D simulations."""
    if len(psi_tail) == 0 or len(g_tail) == 0:
        return "unsettled", np.nan, np.nan

    psi_range = max(psi_tail) - min(psi_tail)
    g_range = max(g_tail) - min(g_tail)
    if psi_range >= eps_psi or g_range >= eps_g:
        return "unsettled", np.nan, np.nan

    psi_mean = float(np.mean(psi_tail))
    g_mean = float(np.mean(g_tail))
    gL = gstar_of_psi(psi_L)
    gH = gstar_of_psi(1.0)

    if abs(psi_mean - psi_L) < eps_psi and abs(g_mean - gL) < eps_g:
        return "low", psi_mean, g_mean
    if abs(psi_mean - 1.0) < eps_psi and abs(g_mean - gH) < eps_g:
        return "high", psi_mean, g_mean
    return "interior_fixed", psi_mean, g_mean

def boundary_basin_disagreement(a2, a1):
    """Low/high basin disagreement, defined only when both models reach boundary basins."""
    boundary = {"low", "high"}
    if a2 in boundary and a1 in boundary:
        return int(a2 != a1)
    return np.nan

def simulate_2d(psi0, g0, theta, dyn, sim, rK_of_psi, Ge_of_psi, gstar_of_psi, n_steps):
    psi_L = theta["psi_L"]
    kappa_c = theta["kappa_c"]
    chi, sigma, omega, mu = dyn["chi"], dyn["sigma"], dyn["omega"], dyn["mu"]
    dt = sim["dt"]
    eps_psi, eps_g, stable_window = sim["eps_psi"], sim["eps_g"], sim["stable_window"]

    psiP = np.clip(psi0 - psi_L, 0.0, 1.0 - psi_L)
    g = max(0.0, g0)

    max_history_len = max(stable_window * 2, 500)
    psi_history = deque(maxlen=max_history_len)
    g_history = deque(maxlen=max_history_len)

    g_series_sparse, t_series_sparse = [], []
    clip_psi_total, clip_g_total, sup_gap_scaled = 0, 0, 0.0

    attractor_type = "unsettled"
    psi_star_hat, g_star_hat, settle_step = np.nan, np.nan, np.nan

    for k in range(1, n_steps + 1):
        psi_now = np.clip(psi_L + psiP, psi_L, 1.0)
        Ge_now = Ge_of_psi(psi_now)
        a_now = kappa_c * psi_now

        g_half = riccati_g_step(g, Ge_now, a_now, chi, 0.5 * dt)
        if not np.isfinite(g_half) or g_half < 0.0:
            g_half = max(0.0, g)

        k1 = rhs_psi(psiP, g, psi_L, rK_of_psi, omega, sigma, mu)
        k2 = rhs_psi(psiP + 0.5 * dt * k1, g_half, psi_L, rK_of_psi, omega, sigma, mu)
        k3 = rhs_psi(psiP + 0.5 * dt * k2, g_half, psi_L, rK_of_psi, omega, sigma, mu)
        k4 = rhs_psi(psiP + dt * k3, g_half, psi_L, rK_of_psi, omega, sigma, mu)

        psiP_new = psiP + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

        psi_mid = np.clip(psi_L + 0.5 * (psiP + psiP_new), psi_L, 1.0)
        Ge_mid = Ge_of_psi(psi_mid)
        a_coeff = kappa_c * psi_mid

        g_new = riccati_g_step(g, Ge_mid, a_coeff, chi, dt)

        psiP_proj = np.clip(psiP_new, 0.0, 1.0 - psi_L)
        if psiP_proj != psiP_new:
            clip_psi_total += 1
        psiP_new = psiP_proj

        if not np.isfinite(g_new) or g_new < 0.0:
            g_new = 0.0
            clip_g_total += 1

        gap_scaled = abs(g_new - gstar_of_psi(psi_L + psiP_new)) / (1.0 + abs(g_new))
        sup_gap_scaled = max(sup_gap_scaled, gap_scaled)

        psiP, g = psiP_new, g_new
        psi_history.append(psi_L + psiP)
        g_history.append(g)

        if k % 10 == 0:
            g_series_sparse.append(g)
            t_series_sparse.append(k * dt)

        if len(psi_history) >= stable_window:
            psi_tail = list(psi_history)[-stable_window:]
            g_tail = list(g_history)[-stable_window:]

            tail_class, psi_mean, g_mean = classify_stable_tail(psi_tail, g_tail, psi_L, gstar_of_psi, eps_psi, eps_g)
            if tail_class != "unsettled":
                settle_step = float(k)
                attractor_type = tail_class
                psi_star_hat, g_star_hat = psi_mean, g_mean
                break

    t_obs = k * dt if k <= n_steps else n_steps * dt

    if attractor_type == "unsettled" and len(psi_history) >= stable_window:
        psi_tail = list(psi_history)[-stable_window:]
        g_tail = list(g_history)[-stable_window:]

        psi_range_min = 0.01 * (1.0 - psi_L)
        g_range_min = 0.001 * max(1.0, abs(np.median(g_tail)))

        if (max(psi_tail) - min(psi_tail)) > psi_range_min and (max(g_tail) - min(g_tail)) > g_range_min:
            dpsi, dg = np.diff(psi_tail), np.diff(g_tail)
            if np.sum(dpsi[:-1] * dpsi[1:] < 0) > stable_window * 0.3 and np.sum(dg[:-1] * dg[1:] < 0) > stable_window * 0.3:
                attractor_type = "cycle_like"

    t_settle = settle_step * dt if np.isfinite(settle_step) else np.nan
    break_diag = compute_break_diagnostics_fixed(g_series_sparse, t_series_sparse)

    return {
        "attractor_type": attractor_type,
        "psi_star_hat": float(psi_star_hat) if np.isfinite(psi_star_hat) else np.nan,
        "g_star_hat": float(g_star_hat) if np.isfinite(g_star_hat) else np.nan,
        "t_settle": float(t_settle) if np.isfinite(t_settle) else np.nan,
        "t_conv": float(t_settle) if np.isfinite(t_settle) else np.nan,
        "t_obs": float(t_obs),
        **break_diag,
        "clip_psi": int(clip_psi_total),
        "clip_g": int(clip_g_total),
        "sup_gap_scaled": float(sup_gap_scaled),
        "converged": bool(attractor_type in ["low", "high", "interior_fixed"]),
        "terminal_basin": attractor_type if attractor_type in ["low", "high"] else "none",
    }

def simulate_1d(psi0, theta, dyn, sim, rK_of_psi, gstar_of_psi, n_steps):
    psi_L = theta["psi_L"]
    dt = sim["dt"]
    eps_psi, eps_g, stable_window = sim["eps_psi"], sim["eps_g"], sim["stable_window"]
    sigma, omega, mu = dyn["sigma"], dyn["omega"], dyn["mu"]

    psiP = np.clip(psi0 - psi_L, 0.0, 1.0 - psi_L)

    max_history_len = max(stable_window * 2, 500)
    psi_history = deque(maxlen=max_history_len)
    g_history = deque(maxlen=max_history_len)

    clip_psi_total, t_conv = 0, np.nan
    attractor_type, basin = "unsettled", "none"
    psi_star_hat, g_star_hat = np.nan, np.nan

    for k in range(1, n_steps + 1):
        k1 = rhs_1d(psiP, psi_L, rK_of_psi, gstar_of_psi, sigma, omega, mu)
        k2 = rhs_1d(psiP + 0.5 * dt * k1, psi_L, rK_of_psi, gstar_of_psi, sigma, omega, mu)
        k3 = rhs_1d(psiP + 0.5 * dt * k2, psi_L, rK_of_psi, gstar_of_psi, sigma, omega, mu)
        k4 = rhs_1d(psiP + dt * k3, psi_L, rK_of_psi, gstar_of_psi, sigma, omega, mu)

        psiP_new = psiP + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

        psiP_proj = np.clip(psiP_new, 0.0, 1.0 - psi_L)
        if psiP_proj != psiP_new:
            clip_psi_total += 1
        psiP = psiP_proj

        psi = psi_L + psiP
        g = gstar_of_psi(psi)

        psi_history.append(psi)
        g_history.append(g)

        if len(psi_history) >= stable_window:
            psi_tail = list(psi_history)[-stable_window:]
            g_tail = list(g_history)[-stable_window:]
            tail_class, psi_mean, g_mean = classify_stable_tail(psi_tail, g_tail, psi_L, gstar_of_psi, eps_psi, eps_g)
            if tail_class != "unsettled":
                t_conv = k * dt
                attractor_type = tail_class
                basin = tail_class if tail_class in ["low", "high"] else "none"
                psi_star_hat, g_star_hat = psi_mean, g_mean
                break

    return {
        "attractor_type": attractor_type,
        "psi_star_hat": float(psi_star_hat) if np.isfinite(psi_star_hat) else np.nan,
        "g_star_hat": float(g_star_hat) if np.isfinite(g_star_hat) else np.nan,
        "clip_psi": int(clip_psi_total),
        "converged": bool(attractor_type in ["low", "high", "interior_fixed"]),
        "t_conv": float(t_conv) if np.isfinite(t_conv) else np.nan,
        "terminal_basin": basin,
    }

def ic_list_block_A(psi0, gstar0):
    s = g_scale(gstar0)
    eps = IC_NEAR_EPS * s
    return [
        ("A_on", float(gstar0)),
        ("A_below", float(max(0.0, gstar0 - eps))),
        ("A_above", float(gstar0 + eps)),
    ]

def ic_list_block_B(psi0, gstar0):
    s = g_scale(gstar0)
    g0_below = float(max(0.0, gstar0 - IC_FAR_BELOW_MULT * s))
    g0_above = float(gstar0 + min(IC_FAR_ABOVE_MULT * s, IC_FAR_ABOVE_CAP_MULT * max(1e-12, abs(gstar0))))
    return [
        ("B_moderate_below", g0_below),
        ("B_moderate_above", g0_above),
    ]

def aggregate_country_block(traj_df):
    if traj_df.empty: return pd.DataFrame()
    grp = traj_df.groupby(["mu_design", "structural_country_idx", "country_idx", "block"], observed=False)

    def _agg(g):
        high_reachable = g["high_boundary_reachable_speed_bound"].astype(bool)
        high_unreachable = ~high_reachable
        return pd.Series({
            "N_traj": len(g),
            "share_low": float(np.mean(g["dynamic_class_2D"] == "low")),
            "share_high": float(np.mean(g["dynamic_class_2D"] == "high")),
            "share_interior_fixed": float(np.mean(g["dynamic_class_2D"] == "interior_fixed")),
            "share_cycle_like": float(np.mean(g["dynamic_class_2D"] == "cycle_like")),
            "share_unsettled": float(np.mean(g["dynamic_class_2D"] == "unsettled")),
            "break_top_quartile_rate": float(np.mean(g["break_top_quartile"])) if "break_top_quartile" in g.columns else np.nan,
            "collapse_rate": float(np.mean(g["collapse_flag"])),
            "growth_drawdown_rate": float(np.mean(g["collapse_flag"])),
            "t_obs_mean": float(np.nanmean(g["t_obs"])) if "t_obs" in g.columns else np.nan,
            "t_obs_median": float(np.nanmedian(g["t_obs"])) if "t_obs" in g.columns else np.nan,
            "psi_travel_capacity_median": float(np.nanmedian(g["psi_travel_capacity"])),
            "threshold_reachable_share": float(np.nanmean(g["threshold_reachable_speed_bound"])) if np.any(np.isfinite(g["threshold_reachable_speed_bound"])) else np.nan,
            "threshold_reachable_comparable_share": float(np.mean(np.isfinite(g["threshold_reachable_speed_bound"]))),
            "high_boundary_reachable_share": float(np.mean(high_reachable)),
            "high_boundary_reachable_high_rate": float(np.mean(g.loc[high_reachable, "dynamic_class_2D"] == "high")) if np.any(high_reachable) else np.nan,
            "high_boundary_unreachable_unsettled_rate": float(np.mean(g.loc[high_unreachable, "dynamic_class_2D"] == "unsettled")) if np.any(high_unreachable) else np.nan,
            "dynamic_class_disagreement_rate": float(np.mean(g["dynamic_class_disagreement"])),
            "boundary_basin_disagreement_rate": float(np.nanmean(g["boundary_basin_disagreement"])) if np.any(np.isfinite(g["boundary_basin_disagreement"])) else np.nan,
            "boundary_basin_comparable_share": float(np.mean(np.isfinite(g["boundary_basin_disagreement"]))),
            "share_none_2D_legacy": float(np.mean(g["basin_2D"] == "none")),
            "share_none_1D": float(np.mean(g["basin_1D"] == "none")),
            "tconv_2D_mean": float(np.nanmean(g.loc[g["conv_2D"], "t_conv_2D"])) if np.any(g["conv_2D"]) else np.nan,
            "tconv_1D_mean": float(np.nanmean(g.loc[g["conv_1D"], "t_conv_1D"])) if np.any(g["conv_1D"]) else np.nan,
            "clip_psi_2D_median": float(np.nanmedian(g["clip_psi_2D"])),
            "clip_g_2D_median": float(np.nanmedian(g["clip_g_2D"])),
            "clip_psi_1D_median": float(np.nanmedian(g["clip_psi_1D"])),
            "sup_gap_scaled_median": float(np.nanmedian(g["sup_gap_scaled"])),
            "sup_gap_scaled_p90": float(np.nanpercentile(g["sup_gap_scaled"], 90)),
        })
    return grp.apply(_agg).reset_index()

def summarize_bins(country_block_df, bins_log10, block_name):
    d = country_block_df[country_block_df["block"] == block_name].copy()
    if d.empty: return pd.DataFrame()

    d["ratio_bin"] = pd.cut(d["log10_ratio"], bins=bins_log10, include_lowest=True)
    grp = d.groupby(["mu_design", "ratio_bin"], observed=False)

    out = grp.agg(
        N=("country_idx", "count"),
        share_interior_fixed_median=("share_interior_fixed", "median"),
        share_cycle_like_median=("share_cycle_like", "median"),
        share_unsettled_median=("share_unsettled", "median"),
        collapse_rate_median=("collapse_rate", "median"),
        growth_drawdown_rate_median=("growth_drawdown_rate", "median"),
        break_top_quartile_rate_median=("break_top_quartile_rate", "median"),
        dynamic_class_disagreement_median=("dynamic_class_disagreement_rate", "median"),
        threshold_reachable_share_median=("threshold_reachable_share", "median"),
        threshold_reachable_comparable_share_median=("threshold_reachable_comparable_share", "median"),
        high_boundary_reachable_share_median=("high_boundary_reachable_share", "median"),
        high_boundary_reachable_high_rate_median=("high_boundary_reachable_high_rate", "median"),
        high_boundary_unreachable_unsettled_rate_median=("high_boundary_unreachable_unsettled_rate", "median"),
        share_high_2D_median=("share_high", "median"),
        share_none_2D_median=("share_none_2D_legacy", "median"),
    ).reset_index()

    out["bin_center"] = out["ratio_bin"].apply(lambda x: 0.5 * (x.left + x.right) if pd.notna(x) else np.nan)
    out["block"] = block_name
    return out

def draw_dyn_params(rng, cfg):
    chi = 10.0 ** rng.uniform(np.log10(cfg["chi_range"][0]), np.log10(cfg["chi_range"][1]))
    sigma = 10.0 ** rng.uniform(np.log10(cfg["sigma_range"][0]), np.log10(cfg["sigma_range"][1]))
    return {
        "chi": float(chi),
        "sigma": float(sigma),
        "omega_mult": rng.uniform(*cfg["omega_mult_range"]),
        "mu": rng.uniform(*cfg["mu_range"])
    }

valid_mu_designs = {"baseline_prior", "threshold_targeted"}
unknown_mu_designs = sorted(set(MU_DESIGNS_TO_RUN) - valid_mu_designs)
if unknown_mu_designs:
    raise ValueError(f"Unknown MU_DESIGNS_TO_RUN entries: {unknown_mu_designs}")

print(f"Generating {N_ACCEPTED_COUNTRIES} structurally admissible countries, then applying mu designs: {MU_DESIGNS_TO_RUN}")

countries = []
dyn_list = []
audit_rows = []
traj_rows = []

rng_theta = np.random.default_rng(SEED)
rng_dyn = np.random.default_rng(SEED + 10)
rng_mu_target = np.random.default_rng(SEED + 1000)

structural_records = []
structural_count = 0
candidate_count = 0

while structural_count < N_ACCEPTED_COUNTRIES and candidate_count < MAX_COUNTRY_CANDIDATES:
    candidate_count += 1
    theta = draw_admissible_primitives(rng_theta, MC_CONFIG)
    objs = build_manifold_objects(theta, MC_CONFIG)
    dyn_base = draw_dyn_params(rng_dyn, MC_CONFIG)
    dyn_base["kappa_c"] = theta["kappa_c"]

    solver_ok_endpoints = objs["solver_ok_endpoints"]
    excluded_bad_share = objs["excluded_bad_share"]
    omega_base = objs["omega_base"]
    omega_ok = np.isfinite(omega_base) and (omega_base > 0.0)
    valid_mu = np.asarray(objs["valid_mask"], dtype=bool)
    threshold_targetable = False
    if len(valid_mu) > 0 and valid_mu[0] and valid_mu[-1] and objs.get("mono_ok", False):
        W_valid = np.asarray(objs["W_star"])[valid_mu]
        W_floor_target = float(W_valid[0])
        W_top_target = float(W_valid[-1])
        threshold_targetable = bool(np.isfinite(W_floor_target) and np.isfinite(W_top_target) and W_top_target > W_floor_target)
    targetability_ok = threshold_targetable if "threshold_targeted" in MU_DESIGNS_TO_RUN else True
    structural_admissible = bool(solver_ok_endpoints and (not excluded_bad_share) and omega_ok and objs["solv_ok"] and objs["mono_ok"] and targetability_ok)

    if structural_admissible:
        structural_records.append({
            "structural_country_idx": structural_count,
            "candidate_idx": candidate_count - 1,
            "theta": theta,
            "objs": objs,
            "dyn_base": dyn_base,
        })
        structural_count += 1
        ratio_msg = dyn_base["chi"] / dyn_base["sigma"]
        if (structural_count == 1 or structural_count % PROGRESS_EVERY_ACCEPTED == 0 or structural_count == N_ACCEPTED_COUNTRIES):
            print(
                f"  [accept:structural] {structural_count}/{N_ACCEPTED_COUNTRIES} countries "
                f"after {candidate_count} candidates | acceptance={structural_count / candidate_count:.3f} | "
                f"log10(chi/sigma)={np.log10(ratio_msg):.2f}"
            )

    if candidate_count % PROGRESS_EVERY_CANDIDATES == 0:
        print(
            f"  [search:structural] candidates={candidate_count} | "
            f"accepted={structural_count}/{N_ACCEPTED_COUNTRIES} | acceptance={structural_count / candidate_count:.3f}"
        )

    structural_threshold_info = classify_static_threshold(objs, dyn_base["mu"])
    audit_rows.append({
        "mu_design": "structural_candidate",
        "structural_country_idx": structural_count - 1 if structural_admissible else -1,
        "candidate_idx": candidate_count - 1,
        "country_idx": structural_count - 1 if structural_admissible else -1,
        "solver_ok_endpoints": bool(solver_ok_endpoints),
        "excluded_bad_share": bool(excluded_bad_share),
        "bad_share": float(objs["bad_share"]),
        "omega_ok": bool(omega_ok),
        "threshold_targetable": bool(threshold_targetable),
        "mu_ok": bool(np.isfinite(dyn_base["mu"])),
        "mu_target_u": np.nan,
        "omega_base": float(omega_base) if np.isfinite(omega_base) else np.nan,
        "omega": np.nan,
        "mu_over_omega": np.nan,
        "deriv_ok": bool(objs["deriv_ok"]),
        "solv_ok": bool(objs["solv_ok"]),
        "gstar_ok": bool(objs["gstar_ok"]),
        "mono_ok": bool(objs["mono_ok"]),
        "admissible": structural_admissible,
        "chi": dyn_base["chi"], "sigma": dyn_base["sigma"], "omega_mult": dyn_base["omega_mult"], "mu": dyn_base["mu"],
        "ratio": dyn_base["chi"] / dyn_base["sigma"], "log10_ratio": float(np.log10(dyn_base["chi"] / dyn_base["sigma"])),
        "psi_travel_capacity": dyn_base["sigma"] * T_FIXED,
        **structural_threshold_info,
        "psi_L": theta["psi_L"], "s_N": theta["s_N"], "Gamma": theta["Gamma"], "k": theta["k"],
        "kappa_c": theta["kappa_c"], "rho_bar": theta["rho_bar"], "a": theta["a"], "b": theta["b"],
        "c_lambda": theta["c_lambda"], "eta_lambda": theta["eta_lambda"], "c_ell": theta["c_ell"], "eta_ell": theta["eta_ell"],
    })

if structural_count < N_ACCEPTED_COUNTRIES:
    raise RuntimeError(
        f"Failed to generate {N_ACCEPTED_COUNTRIES} structurally admissible countries "
        f"within {MAX_COUNTRY_CANDIDATES} candidates."
    )

print("Expanding shared structural sample across mu designs...")
for rec in structural_records:
    structural_idx = rec["structural_country_idx"]
    theta = rec["theta"]
    objs = rec["objs"]
    dyn_base = rec["dyn_base"]
    omega_base = objs["omega_base"]
    omega_val = float(max(MC_CONFIG["OMEGA_MIN"], dyn_base["omega_mult"] * omega_base))
    ratio = dyn_base["chi"] / dyn_base["sigma"]
    log10_ratio = float(np.log10(ratio))
    psi_travel_capacity = dyn_base["sigma"] * T_FIXED

    for mu_design_current in MU_DESIGNS_TO_RUN:
        dyn = dict(dyn_base)
        dyn["mu_design"] = mu_design_current
        dyn["mu_target_u"] = np.nan

        if mu_design_current == "threshold_targeted":
            valid_mu = np.asarray(objs["valid_mask"], dtype=bool)
            W_valid = np.asarray(objs["W_star"])[valid_mu]
            W_floor_mu = float(W_valid[0])
            W_top_mu = float(W_valid[-1])
            if not (np.isfinite(W_floor_mu) and np.isfinite(W_top_mu) and W_top_mu > W_floor_mu):
                raise RuntimeError(f"Shared structural country {structural_idx} is not threshold-targetable.")
            u_mu = rng_mu_target.uniform(*THRESHOLD_TARGET_U_RANGE)
            dyn["mu"] = W_floor_mu + u_mu * (W_top_mu - W_floor_mu)
            dyn["mu_target_u"] = float(u_mu)

        mu_ok = bool(np.isfinite(dyn["mu"]))
        threshold_info = classify_static_threshold(objs, dyn["mu"])
        admissible_design = bool(mu_ok)
        mu_over_omega = dyn["mu"] / omega_val if mu_ok else np.nan

        dyn["omega"] = omega_val
        dyn["country_idx"] = structural_idx
        dyn["structural_country_idx"] = structural_idx
        dyn["psi_travel_capacity"] = psi_travel_capacity
        dyn.update(threshold_info)

        countries.append(theta)
        dyn_list.append(dyn)

        audit_rows.append({
            "mu_design": dyn["mu_design"],
            "structural_country_idx": structural_idx,
            "candidate_idx": rec["candidate_idx"],
            "country_idx": structural_idx,
            "solver_ok_endpoints": bool(objs["solver_ok_endpoints"]),
            "excluded_bad_share": bool(objs["excluded_bad_share"]),
            "bad_share": float(objs["bad_share"]),
            "omega_ok": True,
            "threshold_targetable": True,
            "mu_ok": bool(mu_ok),
            "mu_target_u": dyn["mu_target_u"],
            "omega_base": float(omega_base),
            "omega": omega_val,
            "mu_over_omega": mu_over_omega,
            "deriv_ok": bool(objs["deriv_ok"]),
            "solv_ok": bool(objs["solv_ok"]),
            "gstar_ok": bool(objs["gstar_ok"]),
            "mono_ok": bool(objs["mono_ok"]),
            "admissible": admissible_design,
            "chi": dyn["chi"], "sigma": dyn["sigma"], "omega_mult": dyn["omega_mult"], "mu": dyn["mu"],
            "ratio": ratio, "log10_ratio": log10_ratio, "psi_travel_capacity": psi_travel_capacity,
            **threshold_info,
            "psi_L": theta["psi_L"], "s_N": theta["s_N"], "Gamma": theta["Gamma"], "k": theta["k"],
            "kappa_c": theta["kappa_c"], "rho_bar": theta["rho_bar"], "a": theta["a"], "b": theta["b"],
            "c_lambda": theta["c_lambda"], "eta_lambda": theta["eta_lambda"], "c_ell": theta["c_ell"], "eta_ell": theta["eta_ell"],
        })

print("\nSimulating accepted country trajectories for all mu designs...")
for i, theta in enumerate(countries):
    objs = build_manifold_objects(theta, MC_CONFIG)
    rK_of_psi, Ge_of_psi, gstar_of_psi = make_grid_lookups(objs, theta)

    dyn = dict(dyn_list[i])
    mu_design_current = dyn["mu_design"]
    country_idx = int(dyn["country_idx"])
    structural_country_idx = int(dyn["structural_country_idx"])
    ratio = dyn["chi"] / dyn["sigma"]
    log10_ratio = float(np.log10(ratio))

    T_i = SIM_CONFIG["T_fixed"]
    n_steps_i = int(np.ceil(T_i / SIM_CONFIG["dt"]))
    psi_travel_capacity = dyn["sigma"] * T_i

    if (country_idx == 0 or (country_idx + 1) % PROGRESS_EVERY_COUNTRIES == 0 or (i + 1) == len(countries)):
        expected_traj = MC_CONFIG["psi0_grid_n"] * 5
        print(
            f"  [simulate:{mu_design_current}] country {country_idx + 1}/{N_ACCEPTED_COUNTRIES} "
            f"| global {i + 1}/{len(countries)} | "
            f"log10(chi/sigma)={log10_ratio:.2f} | T={T_i:.1f} | "
            f"steps={n_steps_i} | trajectories={expected_traj}"
        )

    psi0_grid = np.linspace(theta["psi_L"], 1.0, MC_CONFIG["psi0_grid_n"])

    for psi0 in psi0_grid:
        gstar0 = gstar_of_psi(psi0)
        threshold_hat = dyn.get("psi_threshold_hat", np.nan)
        has_interior_threshold = bool(dyn.get("interior_threshold", False)) and np.isfinite(threshold_hat)
        threshold_gap_from_psi0 = abs(threshold_hat - psi0) if has_interior_threshold else np.nan
        threshold_reachable_speed_bound = float(threshold_gap_from_psi0 <= psi_travel_capacity) if has_interior_threshold else np.nan
        high_boundary_gap_from_psi0 = 1.0 - psi0
        high_boundary_reachable_speed_bound = bool(high_boundary_gap_from_psi0 <= psi_travel_capacity)

        for ic_type, g0 in ic_list_block_A(psi0, gstar0):
            out2 = simulate_2d(psi0, g0, theta, dyn, SIM_CONFIG, rK_of_psi, Ge_of_psi, gstar_of_psi, n_steps_i)
            out1 = simulate_1d(psi0, theta, dyn, SIM_CONFIG, rK_of_psi, gstar_of_psi, n_steps_i)

            traj_rows.append({
                "mu_design": mu_design_current, "structural_country_idx": structural_country_idx, "country_idx": country_idx, "block": "A", "ic_type": ic_type, "psi0": float(psi0), "g0": float(g0),
                "chi": dyn["chi"], "sigma": dyn["sigma"], "omega": dyn["omega"], "mu": dyn["mu"], "mu_target_u": dyn["mu_target_u"], "mu_over_omega": dyn["mu"] / dyn["omega"],
                "ratio": ratio, "log10_ratio": log10_ratio,
                "threshold_class": dyn.get("threshold_class", "unclassified"),
                "psi_threshold_hat": dyn.get("psi_threshold_hat", np.nan),
                "interior_threshold": dyn.get("interior_threshold", False),
                "psi_travel_capacity": psi_travel_capacity,
                "threshold_gap_from_psi0": threshold_gap_from_psi0,
                "threshold_reachable_speed_bound": threshold_reachable_speed_bound,
                "high_boundary_gap_from_psi0": high_boundary_gap_from_psi0,
                "high_boundary_reachable_speed_bound": high_boundary_reachable_speed_bound,
                "dynamic_class_2D": out2["attractor_type"], "psi_star_2D": out2["psi_star_hat"], "g_star_2D": out2["g_star_hat"],
                "t_settle_2D": out2["t_settle"], "t_obs": out2["t_obs"], "break_score_mean": out2["break_score_mean"],
                "break_score_var": out2["break_score_var"], "t_break_hat": out2["t_break_hat"], "collapse_flag": out2["collapse_flag"],
                "t_collapse_hat": out2["t_collapse_hat"], "dynamic_class_1D": out1["attractor_type"], "psi_star_1D": out1["psi_star_hat"], "g_star_1D": out1["g_star_hat"],
                "basin_1D": out1["terminal_basin"], "conv_1D": out1["converged"],
                "t_conv_1D": out1["t_conv"], "clip_psi_1D": out1["clip_psi"], "basin_2D": out2["terminal_basin"],
                "conv_2D": out2["converged"], "t_conv_2D": out2["t_conv"],
                "dynamic_class_disagreement": int(out2["attractor_type"] != out1["attractor_type"]),
                "boundary_basin_disagreement": boundary_basin_disagreement(out2["attractor_type"], out1["attractor_type"]),
                "clip_psi_2D": out2["clip_psi"], "clip_g_2D": out2["clip_g"], "T_horizon": T_i, "sup_gap_scaled": out2["sup_gap_scaled"],
            })

        for ic_type, g0 in ic_list_block_B(psi0, gstar0):
            out2 = simulate_2d(psi0, g0, theta, dyn, SIM_CONFIG, rK_of_psi, Ge_of_psi, gstar_of_psi, n_steps_i)
            out1 = simulate_1d(psi0, theta, dyn, SIM_CONFIG, rK_of_psi, gstar_of_psi, n_steps_i)

            traj_rows.append({
                "mu_design": mu_design_current, "structural_country_idx": structural_country_idx, "country_idx": country_idx, "block": "B", "ic_type": ic_type, "psi0": float(psi0), "g0": float(g0),
                "chi": dyn["chi"], "sigma": dyn["sigma"], "omega": dyn["omega"], "mu": dyn["mu"], "mu_target_u": dyn["mu_target_u"], "mu_over_omega": dyn["mu"] / dyn["omega"],
                "ratio": ratio, "log10_ratio": log10_ratio,
                "threshold_class": dyn.get("threshold_class", "unclassified"),
                "psi_threshold_hat": dyn.get("psi_threshold_hat", np.nan),
                "interior_threshold": dyn.get("interior_threshold", False),
                "psi_travel_capacity": psi_travel_capacity,
                "threshold_gap_from_psi0": threshold_gap_from_psi0,
                "threshold_reachable_speed_bound": threshold_reachable_speed_bound,
                "high_boundary_gap_from_psi0": high_boundary_gap_from_psi0,
                "high_boundary_reachable_speed_bound": high_boundary_reachable_speed_bound,
                "dynamic_class_2D": out2["attractor_type"], "psi_star_2D": out2["psi_star_hat"], "g_star_2D": out2["g_star_hat"],
                "t_settle_2D": out2["t_settle"], "t_obs": out2["t_obs"], "break_score_mean": out2["break_score_mean"],
                "break_score_var": out2["break_score_var"], "t_break_hat": out2["t_break_hat"], "collapse_flag": out2["collapse_flag"],
                "t_collapse_hat": out2["t_collapse_hat"], "dynamic_class_1D": out1["attractor_type"], "psi_star_1D": out1["psi_star_hat"], "g_star_1D": out1["g_star_hat"],
                "basin_1D": out1["terminal_basin"], "conv_1D": out1["converged"],
                "t_conv_1D": out1["t_conv"], "clip_psi_1D": out1["clip_psi"], "basin_2D": out2["terminal_basin"],
                "conv_2D": out2["converged"], "t_conv_2D": out2["t_conv"],
                "dynamic_class_disagreement": int(out2["attractor_type"] != out1["attractor_type"]),
                "boundary_basin_disagreement": boundary_basin_disagreement(out2["attractor_type"], out1["attractor_type"]),
                "clip_psi_2D": out2["clip_psi"], "clip_g_2D": out2["clip_g"], "T_horizon": T_i, "sup_gap_scaled": out2["sup_gap_scaled"],
            })

df_audit = pd.DataFrame(audit_rows)
df_traj = pd.DataFrame(traj_rows)

if not df_traj.empty and "break_score_mean" in df_traj.columns:
    global_break_cutoff = df_traj["break_score_mean"].quantile(0.75)
    df_traj["break_top_quartile"] = df_traj["break_score_mean"] > global_break_cutoff
else:
    df_traj["break_top_quartile"] = False

df_country_block = aggregate_country_block(df_traj)
if not df_country_block.empty:
    df_country_block = df_country_block.merge(
        df_audit.loc[
            df_audit["admissible"],
            ["mu_design", "structural_country_idx", "country_idx", "ratio", "log10_ratio", "psi_travel_capacity", "mu_target_u", "threshold_class", "psi_threshold_hat", "interior_threshold", "W_floor", "W_top"]
        ].drop_duplicates(),
        on=["mu_design", "structural_country_idx", "country_idx"], how="left"
    )

df_bins_A = summarize_bins(df_country_block, MC_CONFIG["ratio_bins_log10"], "A")
df_bins_B = summarize_bins(df_country_block, MC_CONFIG["ratio_bins_log10"], "B")
df_audit_design = df_audit[df_audit["mu_design"].isin(MU_DESIGNS_TO_RUN)].copy()
df_audit_structural = df_audit[df_audit["mu_design"] == "structural_candidate"].copy()

summary_rows = []
total_candidates = len(df_audit_structural)
accepted_count = int(df_audit_structural["admissible"].sum()) if total_candidates else 0
summary_rows.extend([
    {"metric": "structural_candidates_evaluated", "value": total_candidates},
    {"metric": "accepted_structural_countries", "value": accepted_count},
    {"metric": "country_design_records", "value": len(df_audit_design)},
    {"metric": "acceptance_rate", "value": accepted_count / total_candidates if total_candidates else np.nan},
    {"metric": "rejected_candidates", "value": total_candidates - accepted_count},
    {"metric": "solver_ok_endpoint_share", "value": float(df_audit_structural["solver_ok_endpoints"].mean()) if total_candidates else np.nan},
    {"metric": "bad_share_exclusion_rate", "value": float(df_audit_structural["excluded_bad_share"].mean()) if total_candidates else np.nan},
    {"metric": "omega_ok_share", "value": float(df_audit_structural["omega_ok"].mean()) if total_candidates else np.nan},
    {"metric": "threshold_targetable_share", "value": float(df_audit_structural["threshold_targetable"].mean()) if total_candidates else np.nan},
    {"metric": "solv_ok_share", "value": float(df_audit_structural["solv_ok"].mean()) if total_candidates else np.nan},
    {"metric": "solvency_exclusion_rate", "value": float((~df_audit_structural["solv_ok"]).mean()) if total_candidates else np.nan},
    {"metric": "gstar_ok_share", "value": float(df_audit_structural["gstar_ok"].mean()) if total_candidates else np.nan},
    {"metric": "mono_ok_share", "value": float(df_audit_structural["mono_ok"].mean()) if total_candidates else np.nan},
    {"metric": "deriv_ok_share", "value": float(df_audit_structural["deriv_ok"].mean()) if total_candidates else np.nan},
])
for cls, n_cls in df_audit_structural["threshold_class"].value_counts(dropna=False).items():
    summary_rows.append({"metric": f"threshold_class_all_{cls}", "value": int(n_cls)})
accepted_threshold_counts = df_audit_structural.loc[df_audit_structural["admissible"], "threshold_class"].value_counts(dropna=False)
for cls, n_cls in accepted_threshold_counts.items():
    summary_rows.append({"metric": f"threshold_class_accepted_{cls}", "value": int(n_cls)})
for design_name, dsum in df_audit_design.groupby("mu_design", observed=False):
    total_d = len(dsum)
    accepted_d = int(dsum["admissible"].sum()) if total_d else 0
    summary_rows.extend([
        {"metric": f"{design_name}_total_candidates_evaluated", "value": total_d},
        {"metric": f"{design_name}_accepted_countries", "value": accepted_d},
        {"metric": f"{design_name}_acceptance_rate", "value": accepted_d / total_d if total_d else np.nan},
        {"metric": f"{design_name}_solv_ok_share", "value": float(dsum["solv_ok"].mean()) if total_d else np.nan},
        {"metric": f"{design_name}_mu_ok_share", "value": float(dsum["mu_ok"].mean()) if total_d else np.nan},
        {"metric": f"{design_name}_threshold_targetable_share", "value": float(dsum["threshold_targetable"].mean()) if total_d else np.nan},
        {"metric": f"{design_name}_mono_ok_share", "value": float(dsum["mono_ok"].mean()) if total_d else np.nan},
    ])
    for cls, n_cls in dsum["threshold_class"].value_counts(dropna=False).items():
        summary_rows.append({"metric": f"{design_name}_threshold_class_all_{cls}", "value": int(n_cls)})
    for cls, n_cls in dsum.loc[dsum["admissible"], "threshold_class"].value_counts(dropna=False).items():
        summary_rows.append({"metric": f"{design_name}_threshold_class_accepted_{cls}", "value": int(n_cls)})
df_admissibility_summary = pd.DataFrame(summary_rows)

paths = {
    "admissibility_summary": save_df(df_admissibility_summary, "admissibility_summary"),
    "audit": save_df(df_audit, "audit"),
    "traj": save_df(df_traj, "traj"),
    "country_block": save_df(df_country_block, "country_block"),
    "bins_A": save_df(df_bins_A, "bins_blockA"),
    "bins_B": save_df(df_bins_B, "bins_blockB")
}

def save_plot_scatter_with_binned(df, x_col, y_col, bins_df, y_bin_col, title, out_stub):
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    ax.scatter(df[x_col], df[y_col], s=15, c="0.6", alpha=0.65, edgecolors="none")
    bb = bins_df.dropna(subset=["bin_center", y_bin_col])
    if not bb.empty:
        ax.plot(bb["bin_center"], bb[y_bin_col], color="0.2", linewidth=2.0)
    ax.set_title(title)
    ax.set_xlabel(r"$\log_{10}(\chi/\sigma)$")
    ax.set_ylabel(y_col)
    ax.grid(alpha=0.25, color="0.8")
    fig.tight_layout()
    png = os.path.join(OUT_DIR, f"{out_stub}_{RUN_TS}.png")
    pdf = os.path.join(OUT_DIR, f"{out_stub}_{RUN_TS}.pdf")
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    return png, pdf

def plot_block(block_name, mu_design, df_cb, df_bins):
    d = df_cb[(df_cb["block"] == block_name) & (df_cb["mu_design"] == mu_design)].copy()
    df_bins = df_bins[df_bins["mu_design"] == mu_design].copy()
    if d.empty or df_bins.empty: return {}

    out = {}
    design_stub = str(mu_design).replace(" ", "_")
    design_title = str(mu_design).replace("_", " ")

    png, pdf = save_plot_scatter_with_binned(d, "log10_ratio", "share_interior_fixed", df_bins, "share_interior_fixed_median", f"Interior Fixed Attractors vs log10(chi/sigma): {design_title}, Block {block_name}", f"interior_fixed_vs_ratio_{design_stub}_block{block_name}")
    out["interior_fixed_png"] = png
    out["interior_fixed_pdf"] = pdf

    png, pdf = save_plot_scatter_with_binned(d, "log10_ratio", "growth_drawdown_rate", df_bins, "growth_drawdown_rate_median", f"Growth Drawdown Diagnostic Rate vs log10(chi/sigma): {design_title}, Block {block_name}", f"growth_drawdown_vs_ratio_{design_stub}_block{block_name}")
    out["growth_drawdown_png"] = png
    out["growth_drawdown_pdf"] = pdf

    png, pdf = save_plot_scatter_with_binned(d, "log10_ratio", "break_top_quartile_rate", df_bins, "break_top_quartile_rate_median", f"Break Top Quartile Rate vs log10(chi/sigma): {design_title}, Block {block_name}", f"break_topq_vs_ratio_{design_stub}_block{block_name}")
    out["break_topq_png"] = png
    out["break_topq_pdf"] = pdf

    png, pdf = save_plot_scatter_with_binned(d, "log10_ratio", "dynamic_class_disagreement_rate", df_bins, "dynamic_class_disagreement_median", f"Dynamic Class Disagreement Rate vs log10(chi/sigma): {design_title}, Block {block_name}", f"dynamic_class_disagreement_vs_ratio_{design_stub}_block{block_name}")
    out["dynamic_class_disagreement_png"] = png
    out["dynamic_class_disagreement_pdf"] = pdf

    return out

plots = {
    design: {
        "A": plot_block("A", design, df_country_block, df_bins_A),
        "B": plot_block("B", design, df_country_block, df_bins_B),
    }
    for design in MU_DESIGNS_TO_RUN
}

for design in MU_DESIGNS_TO_RUN:
    for block in ["A", "B"]:
        for k, v in plots[design][block].items():
            paths[f"plot_{design}_{block}_{k}"] = v

config_dump = {
    "MC_NAME": MC_NAME, "RUN_TS": RUN_TS, "SEED": SEED,
    "RUN_PARAMETERS": {
        "N_ACCEPTED_COUNTRIES": N_ACCEPTED_COUNTRIES,
        "MAX_COUNTRY_CANDIDATES": MAX_COUNTRY_CANDIDATES,
        "PSI_GRID_N": PSI_GRID_N,
        "PSI0_GRID_N": PSI0_GRID_N,
        "T_FIXED": T_FIXED,
        "DT": DT,
        "CHI_RANGE": CHI_RANGE,
        "SIGMA_RANGE": SIGMA_RANGE,
        "OMEGA_MULT_RANGE": OMEGA_MULT_RANGE,
        "MU_RANGE": MU_RANGE,
        "MU_DESIGNS_TO_RUN": MU_DESIGNS_TO_RUN,
        "THRESHOLD_TARGET_U_RANGE": THRESHOLD_TARGET_U_RANGE,
        "PROGRESS_EVERY_CANDIDATES": PROGRESS_EVERY_CANDIDATES,
        "PROGRESS_EVERY_ACCEPTED": PROGRESS_EVERY_ACCEPTED,
        "PROGRESS_EVERY_COUNTRIES": PROGRESS_EVERY_COUNTRIES,
    },
    "MC_CONFIG": MC_CONFIG, "SIM_CONFIG": SIM_CONFIG,
    "IC_NEAR_EPS": IC_NEAR_EPS, "IC_FAR_BELOW_MULT": IC_FAR_BELOW_MULT, "IC_FAR_ABOVE_MULT": IC_FAR_ABOVE_MULT, "IC_FAR_ABOVE_CAP_MULT": IC_FAR_ABOVE_CAP_MULT,
    "plots": plots,
    "platform": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "matplotlib": plt.matplotlib.__version__}
}
config_path = os.path.join(OUT_DIR, "config_dump.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_dump, f, indent=2)
paths["config_dump"] = config_path

print("\n" + "=" * 100)
print(f"mc_block2_threshold_dynamics - Dynamic Stability MC complete | MC_NAME={MC_NAME} | RUN_TS={RUN_TS}")
print("=" * 100)
print(f"N_structural={MC_CONFIG['N_COUNTRIES']} | designs={MU_DESIGNS_TO_RUN} | country_design_records={len(df_audit_design)}")
if not df_audit.empty:
    print(
        f"Structural candidates evaluated={len(df_audit_structural)} | acceptance={df_audit_structural['admissible'].mean():.3f} | "
        f"solv_ok={df_audit_structural['solv_ok'].mean():.3f} | solvency_excluded={(~df_audit_structural['solv_ok']).mean():.3f}"
    )
    print("Static threshold classes among accepted countries by mu design:")
    print(pd.crosstab(df_audit_design.loc[df_audit_design["admissible"], "mu_design"], df_audit_design.loc[df_audit_design["admissible"], "threshold_class"]).to_string())
print("\nSaved outputs:")
for k, v in paths.items():
    print(f"  {k:28s} -> {v}")
print("=" * 100)
